# IndicCrimLawLLM — QLoRA fine-tune of Qwen 2.5 7B Instruct

Kaggle T4 (16 GB VRAM) target. Trains a LoRA adapter on the v0.2
instruction dataset (1,869 pairs from `mapping_qa`,
`section_interpretation`, `refusal`, `bns_transition`).

**Smoke test first** (5 steps, ~5 min) — confirms the stack loads,
the model fits, gradients flow, and the adapter saves. Only after
smoke passes flip `SMOKE_TEST = False` and run the full 3-epoch job
(estimated 18-24h on a single T4).

**Prerequisites in the Kaggle notebook UI:**
1. Settings → Accelerator → **GPU T4 x1**.
2. Settings → Internet → **On** (for `pip install` and `git clone`).
3. Add Data → search and attach the
   `indiccrimlawllm-instruction-v02` dataset (uploaded via
   `scripts/prepare_kaggle_dataset.py` then `kaggle datasets create`).
4. Add-ons → Secrets → add `HF_TOKEN` (HuggingFace token, read scope).

In [ ]:
# Pinned versions for reproducibility on Kaggle. Re-run
# requires kernel restart only if these versions changed.
%pip install --quiet --upgrade \
    'transformers==4.46.3' \
    'peft==0.13.2' \
    'trl==0.12.2' \
    'bitsandbytes==0.44.1' \
    'accelerate==1.1.1' \
    'datasets==3.1.0'

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA device — switch Accelerator to GPU T4 in Kaggle settings.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print('Torch:', torch.__version__)

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

# Kaggle dataset slug — adjust if you used a different slug at upload time.
DATASET_SLUG = 'indiccrimlawllm-instruction-v02'
DATASET_PATH = f'/kaggle/input/{DATASET_SLUG}/instruction_dataset_v0.2.jsonl'
OUTPUT_DIR = '/kaggle/working/output'
REPO_DIR = '/kaggle/working/repo'

assert Path(DATASET_PATH).exists(), f'Dataset not found at {DATASET_PATH} — check the Add Data step.'
print('Dataset:', DATASET_PATH)
print('Output:', OUTPUT_DIR)

## Reproducibility

This notebook trains a QLoRA adapter on Qwen 2.5 7B Instruct using
the indic-criminal-law-llm instruction dataset.

For the production run that produced the released model, set
`REPO_COMMIT` below to the specific commit SHA recorded in the
paper's experimental setup. For experimentation, leave as `'main'`.

- **Default:** `main` (latest)
- **Production:** see `docs/training/training_v1_setup.md` for the pinned SHA

In [ ]:
REPO_URL = 'https://github.com/ChaitanyaKis/indic-criminal-law-llm.git'
REPO_COMMIT = 'main'  # change to specific commit SHA for paper reproducibility

!git clone {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout {REPO_COMMIT}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo:', REPO_DIR, '@', REPO_COMMIT)

In [ ]:
SMOKE_TEST = True  # FLIP TO False FOR THE FULL 3-EPOCH RUN

from scripts.train_qlora import main as train_main

argv = [
    '--dataset', DATASET_PATH,
    '--output-dir', OUTPUT_DIR,
    '--base-model', 'Qwen/Qwen2.5-7B-Instruct',
]
if SMOKE_TEST:
    argv += ['--max-steps', '5']
    print('=== SMOKE TEST: 5 steps ===')
else:
    print('=== FULL RUN: 3 epochs ===')

train_main(argv)

In [ ]:
# Verify the LoRA adapter was saved. The download step in Kaggle's
# UI grabs the contents of /kaggle/working/.
out = Path(OUTPUT_DIR)
for p in sorted(out.iterdir()):
    print(f'{p.name:<40s} {p.stat().st_size:>12,} bytes')
print()
print('Total adapter size (MB):', sum(p.stat().st_size for p in out.iterdir() if p.is_file()) / 1e6)